# Phase 4 — Joint partial-supervision fine-tune (LG-PM, pre-registered family G/H)

**Question (pre-registered before execution):** can a single jointly fine-tuned trunk
(TN3K masks + TN5000 pathology labels, partial supervision) **close the in-domain
fine-tuning gap to whole-image ERM** (Phase 3b test T2) **without importing ERM's
external fragility** (Phase 3c/3d operating-point finding)?

**Design (two autologous arms — arms differ ONLY in the pooling region of the
classification head):**
- **Trunk:** Phase-1 ResNet34-UNet checkpoint, *unfrozen*, fine-tuned with TN3K
  segmentation batches (BCE + Tversky) interleaved with TN5000 classification
  batches (BCE + pos_weight), one shared AdamW/cosine regimen;
- **JOINT-g arm:** R2-free R1p-style lesion head on the trunk's OWN predicted masks
  (soft mask during training, hard mask under the locked R1p conventions at eval);
- **JOINT-img arm:** whole-image GAP head — the multi-task ablation isolating
  *grounding* from *mere multi-tasking*.

**Regimen (protocol v2, 2026-08-21 amendment).** Two phases per arm:
**(W)** trunk-frozen head warmup (AdamW lr 1e-4 / wd 1e-4, cosine T_max=25, 25
epochs, head parameters only) and **(J)** full-trunk joint fine-tune at lr 1e-5
(an order below the ERM regimen), cosine T_max=15, <=15 epochs, patience 4,
seed 42, AMP. Checkpoint selection maximises TN5000-val AUROC SUBJECT TO a
tn3k-val Dice guard (>=0.8066: the pre-registered test-gate constant plus 0.01
margin, val split only — test stays untouched). `set_seed(42)` is the first
statement of each arm's run, so both arms see identical batch streams and
augmentation draws (warmup consumes a fixed, data-independent number of draws
per arm). v2 history: protocol v1 (single-stage joint fine-tune at lr 1e-4,
AUROC-only selection) failed the pre-registered segmentation non-collapse gate
on 2026-08-21 (JOINT-g Dice 0.6686 < 0.7966; triage-isolated mechanism:
fresh-head cls gradients rotated the shared encoder from epoch 1); v2 was
locked before any family-G/H evaluation.

**Pre-registered tests (raw-raw convention; thresholds/temperature from TN5000-val
only):**
- In-domain family **G** (paired bootstrap, 5000 resamples, Bonferroni 0.05/4 = 0.0125):
  G1 `JOINTg − ERM` AUROC (is T2 closed?) · G2 `JOINTg − R1p` AUROC (frozen vs joint
  grounding) · G3 `JOINTg − JOINTimg` AUROC (grounding vs multi-tasking) ·
  G4 `sens@spec90 JOINTg − ERM` (+ McNemar).
- External family **H** (pooled TNUI test+val — the single locked pool from 3d;
  2000 resamples, Bonferroni 0.05/3 = 0.0167): H1 `sens@spec90 JOINTg − ERM`
  (PRIMARY, op-point) · H2 `sens@spec90 JOINTg − R1p` · H3 `AUROC JOINTg − ERM`.
  TNUI-train + DDTI are descriptive rows only (per the locked pooling policy).
- Gap-6 component: selective prediction — coverage vs malignant-FNR curves from
  calibrated JOINT-g probabilities on TN5000-test and pooled TNUI.

**Gates (hard asserts):** TN5000 manifest re-derives exactly 3448/476/950; TN3K train
pool re-derives 2447; ERM sanity |AUROC − 0.9401| <= 0.002 from locked probs; R1p
cache integrity |AUROC − 0.8902| <= 0.002; joint segmentation must not collapse
(TN3K-test Dice >= locked 0.8466 − 0.05, both trunks).

**Required attachments (Kaggle datasets), searched under /kaggle/input AND
/kaggle/working:** raw: `tn3k-data`, `tn5000`, `tnui-2021`; checkpoint:
`best_resnet34_seed42.pt` (Phase 1); Phase 3: `features_val.npz`, `features_test.npz`,
`erm_resnet34_probs.json`, `erm_resnet34_tn5000.pt`, `linear_heads_results.json`;
Phase 3c: `features_tnui_test.npz`, `linear_heads_retrained.json`;
Phase 3d: `features_tnui_val.npz`; Phase 3e: `features_tnui_ddti.npz`,
`features_tnui_tnuitrain.npz`.

**Runtime estimate (T4):** two arms x <=15 epochs, ~55-80 min total + <15 min battery.

## 0. Configuration, cache/checkpoint discovery, protocol constants
All constants fixed pre-registration; mirrored from Phases 1-3 where applicable.

In [ ]:
from pathlib import Path
import json, time, random, hashlib, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from IPython.display import display
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN = DEVICE.type == "cuda"

INPUT_ROOT  = Path("/kaggle/input/datasets")
AUTHOR_ROOT = INPUT_ROOT / "jonbon8397"
PATHS = {
    "tn3k":   AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset" / "tn3k",
    "tn5000": AUTHOR_ROOT / "tn5000" / "TN5000_forReview",
    "tnui":   AUTHOR_ROOT / "tnui-2021" / "TNUI-2021-" / "thyroid_data",
}
DDTI_ROOT = PATHS["tn3k"].parent / "DDTI dataset" / "DDTI"      # Phase-2/3e-verified layout
OUTPUT_ROOT = Path("/kaggle/working/phase4_joint")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def load_ckpt(path):
    # torch >= 2.6 defaults torch.load(..., weights_only=True); our own checkpoints
    # embed numpy scalars, so fall back to full unpickling for trusted artifacts.
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        return torch.load(path, map_location="cpu", weights_only=False)

def find_file(name):
    """Locate a required artifact in /kaggle/working first, then /kaggle/input."""
    hits = sorted(Path("/kaggle/working").rglob(name)) + \
           sorted(Path("/kaggle/input").rglob(name))
    return hits[0] if hits else None

REQUIRED = {
    "ckpt_phase1":  "best_resnet34_seed42.pt",
    "tn5k_fval":    "features_val.npz",
    "tn5k_ftest":   "features_test.npz",
    "erm_probs":    "erm_resnet34_probs.json",
    "erm_ckpt":     "erm_resnet34_tn5000.pt",
    "lin_heads":    "linear_heads_results.json",
    "lin_retr":     "linear_heads_retrained.json",
    "bank_tnui_test":  "features_tnui_test.npz",
    "bank_tnui_val":   "features_tnui_val.npz",
    "bank_tnui_train": "features_tnui_tnuitrain.npz",
    "bank_ddti":       "features_tnui_ddti.npz",
}
ART = {k: find_file(v) for k, v in REQUIRED.items()}
missing = [f"{k} ({v})" for k, v in REQUIRED.items() if ART[k] is None]
assert not missing, ("missing artifacts: " + "; ".join(missing) +
    " - attach the Phase 1/3/3c/3d/3e output folders as Kaggle datasets and re-run.")
for k, v in ART.items():
    print(f"  {k:16s} -> {v}")

# ---- protocol constants (locked pre-registration) ----
IMG_SIZE, BATCH = 384, 32
PRED_THRESH, SEG_EVAL_THRESH = 0.5, 0.60            # eval mask binarisation; locked Phase-1 seg threshold
FEAT_SCALES = ["e3", "e4"]                          # descriptor levels (256+512 = 768-d)
EPOCHS, PATIENCE = 15, 4                            # joint-phase budget (mirrored ERM regimen)
LR, WD = 1e-4, 1e-4                                 # warmup head LR / weight decay (LR mirrored ERM)
LAMBDA_SEG, LAMBDA_CLS = 1.0, 1.0                   # partial-supervision weights
TV_ALPHA, TV_BETA = 0.3, 0.7                        # Tversky (Phase-1 constants)
MASK_THRESH = 127                                   # TN3K mask binarisation
N_BOOT_G, N_BOOT_H = 5000, 2000
ALPHA_G, ALPHA_H = 0.05 / 4, 0.05 / 3
N_BOOT_MAIN = 2000
SUPERVISED_REF = {"TN3K-test": 0.8466, "TNUI-test": 0.7464, "DDTI": 0.7396}   # locked Phase-1 TTA refs
PROTO_REF = {"TNUI-test": 0.7079, "DDTI": 0.6782}                              # locked Phase-2 K=20 zero-shot refs
SEG_GATE_TOL = 0.05
# --- v2 protocol amendment (2026-08-21), two-phase training -----------------------
# v1 (single-stage joint FT, shared LR 1e-4, AUROC-only selection) FAILED the
# pre-registered seg non-collapse gate: the fresh head's cls gradients rotated the
# shared encoder from epoch 1 (seg loss rising while cls loss fell; saved JOINT-g
# TN3K-test Dice 0.6686 < gate 0.7966; harness validated by pristine control 0.8441
# reproducing the locked single-view ref EXACTLY). v2 = (W) trunk-frozen head
# warmup (both tasks near their minima at unfreeze), (J) low-LR joint phase,
# seg-guarded checkpoint selection on tn3k-VAL (test stays untouched). v2 locked
# before any family-G/H evaluation.
LR_JOINT = 1e-5            # joint-phase LR: an order of magnitude below v1/ERM 1e-4
EPOCHS_WARM = 25           # trunk-frozen head-warmup budget (no early stop; fixed RNG draws per arm)
SEG_SELECT_GUARD = SUPERVISED_REF["TN3K-test"] - SEG_GATE_TOL + 0.01   # = 0.8066, applied on tn3k-val
ERM_REF_AUROC, R1P_REF_AUROC, GATE_TOL_AUROC = 0.9401, 0.8902, 0.002           # locked Phase-3 rows
print("device:", DEVICE, "| output:", OUTPUT_ROOT)

## 1. Manifests — TN3K training pool + TN5000 hard gate; TN3K test pairs
Byte-identical re-derivation of the Phase 1-3 manifests. Gates: TN3K train pool = 2447;
TN5000 splits = exactly {train 3448, val 476, test 950} (else the locked comparisons are void).

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

# ---------- TN3K (segmentation loss pool + test gate; identical rules to Phases 1-3) ----------
def pair_images_masks(image_dir, mask_dir, dataset, split):
    imgs = {p.stem: p for p in image_files(image_dir)}
    msks = {p.stem: p for p in image_files(mask_dir)}
    common = sorted(set(imgs) & set(msks))
    return pd.DataFrame([{"dataset": dataset, "split": split, "id": k,
                          "image": str(imgs[k]), "mask": str(msks[k])} for k in common])

tn3k = pd.concat([
    pair_images_masks(PATHS["tn3k"]/"trainval-image", PATHS["tn3k"]/"trainval-mask", "TN3K", "trainval"),
    pair_images_masks(PATHS["tn3k"]/"test-image",     PATHS["tn3k"]/"test-mask",     "TN3K", "test"),
], ignore_index=True)
tn3k["sha256"] = tn3k["image"].apply(lambda p: sha256_file(p) if Path(p).exists() else None)
tn3k["_p"] = tn3k["split"].map({"trainval": 0, "test": 1})
tn3k = tn3k.sort_values(["_p", "id"], kind="stable").drop_duplicates("sha256", keep="first").drop(columns="_p")
rng = np.random.RandomState(SEED)
tv = tn3k[tn3k.split == "trainval"].copy()
perm = rng.permutation(len(tv))
val_ids = set(tv.iloc[perm[: int(round(0.15 * len(tv)))]].id)      # same val as Phases 1-3
tv["split"] = np.where(tv.id.isin(val_ids), "val", "train")
tn3k_train = tv[tv.split == "train"].reset_index(drop=True)
tn3k_val   = tv[tv.split == "val"].reset_index(drop=True)
tn3k_test  = tn3k[tn3k.split == "test"].reset_index(drop=True)
assert len(tn3k_train) == 2447, f"TN3K train pool {len(tn3k_train)} != locked 2447 - manifest drift!"
print(f"TN3K train pool {len(tn3k_train)} (gate OK) | val {len(tn3k_val)} | test {len(tn3k_test)}")

# ---------- TN5000 (classification; pathology labels) ----------
import xml.etree.ElementTree as ET
def parse_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    boxes, names = [], []
    for ob in root.findall("object"):
        nm = (ob.findtext("name") or "").strip().lower()
        bb = ob.find("bndbox")
        if bb is None:
            continue
        try:
            box = [float(bb.findtext(t)) for t in ("xmin", "ymin", "xmax", "ymax")]
        except (TypeError, ValueError):
            continue
        boxes.append(box); names.append(nm)
    return boxes, names

BENIGN, MALIGNANT = {"0", "benign"}, {"1", "malignant"}
def xml_to_row(xml_path):
    boxes, names = parse_xml(xml_path)
    if not boxes:
        return None
    mal = [i for i, n in enumerate(names) if n in MALIGNANT and n not in BENIGN]
    label = 1 if mal else 0
    use = [boxes[i] for i in (mal if mal else range(len(boxes)))]
    x0 = min(b[0] for b in use); y0 = min(b[1] for b in use)
    x1 = max(b[2] for b in use); y1 = max(b[3] for b in use)
    return {"label": label, "n_obj": len(boxes), "box": [x0, y0, x1, y1]}

t5_img = {p.stem: p for p in image_files(PATHS["tn5000"]/"JPEGImages")}
xml_dir = PATHS["tn5000"]/"Annotations"
xmls = {p.stem: p for p in xml_dir.glob("*.xml")} if xml_dir.exists() else {}
common = sorted(set(t5_img) & set(xmls))
assert common, "TN5000: no JPEG/XML name overlap - check mount paths."
rows, skipped = [], 0
for k in common:
    r = xml_to_row(xmls[k])
    if r is None:
        skipped += 1; continue
    rows.append({"dataset": "TN5000", "id": k, "image": str(t5_img[k]), **r})
tn5000 = pd.DataFrame(rows)
print(f"TN5000: {len(tn5000)} usable (skipped {skipped}) | prevalence {tn5000.label.mean():.3f}")

main_dir = PATHS["tn5000"]/"ImageSets"/"Main"
def read_split_file(stem):
    for cand in (main_dir/f"{stem}.txt", main_dir/f"{stem.upper()}.txt"):
        if cand.exists():
            ids = []
            for ln in cand.read_text().splitlines():
                ln = ln.strip()
                if ln:
                    ids.append(Path(ln.split()[0]).stem)
            return ids, str(cand)
    return None, None
tn5000["split"] = None
_used = {}
for stem in ["train", "val", "test"]:
    ids, src = read_split_file(stem)
    assert ids is not None, f"official TN5000 split file for {stem} missing (no fallback permitted)"
    tn5000.loc[tn5000.id.isin(ids), "split"] = stem; _used[stem] = src
print("official split files:", {k: Path(v).name for k, v in _used.items()})
assert int(tn5000.split.isna().sum()) == 0, "TN5000 rows not covered by the split files"
tn5000 = tn5000.reset_index(drop=True)

tn5000["sha256"] = tn5000["image"].apply(sha256_file)
tn5000["_p"] = tn5000["split"].map({"train": 0, "val": 1, "test": 2})
before = len(tn5000)
tn5000 = tn5000.sort_values(["_p", "id"], kind="stable").drop_duplicates("sha256", keep="first")
tn5000 = tn5000.drop(columns="_p").reset_index(drop=True)
print(f"TN5000 dedup: {before} -> {len(tn5000)} (dropped {before - len(tn5000)})")
ov = set(tn5000.sha256) & set(tn3k_train.sha256)
if ov:
    tn5000 = tn5000[~tn5000.sha256.isin(ov)].reset_index(drop=True)
print(f"cross-dataset overlap TN5000 vs TN3K-train: {len(ov)}")

SPLITS = {s: tn5000[tn5000.split == s].reset_index(drop=True) for s in ["train", "val", "test"]}
_sizes = {s: len(d) for s, d in SPLITS.items()}
assert _sizes == {"train": 3448, "val": 476, "test": 950}, \
    f"TN5000 split drift {_sizes} != locked {{train: 3448, val: 476, test: 950}}"
for s, d in SPLITS.items():
    print(f"  {s:5s} n={len(d):5d}  malignant={int(d.label.sum()):4d} ({d.label.mean():.3f})")
print("MANIFEST GATE: TN5000 split sizes re-derived exactly (3448/476/950).")

## 2. Loaders, augmentation, and deterministic batch streams
Segmentation batches get joint image+mask augmentation (flip, ±10° rotation, gain);
classification batches get the ERM-matched image augmentation (flip, ±7° rotation,
gain 0.8-1.2). Batch streams are numpy-permutation driven: after `set_seed(42)` each
arm consumes the identical stream, so the arms are autologous except for the head.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def read_gray(p):
    return np.asarray(Image.open(p).convert("L"))

def letterbox_image(img, size=IMG_SIZE):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    ri = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    t, l = (size - nh) // 2, (size - nw) // 2
    ci = np.zeros((size, size), np.float32); ci[t:t + nh, l:l + nw] = ri / 255.0
    return ci

def letterbox_mask(msk_raw, size=IMG_SIZE):
    h, w = msk_raw.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    rm = cv2.resize(msk_raw, (nw, nh), interpolation=cv2.INTER_NEAREST)
    t, l = (size - nh) // 2, (size - nw) // 2
    cm = np.zeros((size, size), np.uint8); cm[t:t + nh, l:l + nw] = (rm > 0).astype(np.uint8)
    return cm

def to_tensor(img):
    x = torch.from_numpy(np.stack([img, img, img])).float()
    return (x - MEAN) / STD

def dice_np(pred, gt):
    pred = (pred > 0).astype(np.float64).ravel(); gt = (gt > 0).astype(np.float64).ravel()
    den = pred.sum() + gt.sum()
    return 1.0 if den == 0 else float(2.0 * (pred * gt).sum() / den)

def load_tn3k_item(row):
    img = letterbox_image(read_gray(row["image"]).astype(np.float32))
    msk = letterbox_mask((read_gray(row["mask"]) >= MASK_THRESH).astype(np.uint8))
    return img, msk

def seg_aug(img, msk):
    """Joint flip / ±10° rotation (shared geometry), gain on the image only."""
    if np.random.rand() < 0.5:
        img = np.ascontiguousarray(img[:, ::-1]); msk = np.ascontiguousarray(msk[:, ::-1])
    if np.random.rand() < 0.5:
        ang = np.random.uniform(-10, 10)
        M = cv2.getRotationMatrix2D((IMG_SIZE/2, IMG_SIZE/2), ang, 1.0)
        img = cv2.warpAffine(img, M, (IMG_SIZE, IMG_SIZE), flags=cv2.INTER_LINEAR)
        msk = cv2.warpAffine(msk, M, (IMG_SIZE, IMG_SIZE), flags=cv2.INTER_NEAREST)
    if np.random.rand() < 0.5:
        img = np.clip(img * np.random.uniform(0.8, 1.2), 0, 1)
    return img.astype(np.float32), msk.astype(np.uint8)

def cls_aug(img):
    """ERM-matched image augmentation (flip / ±7° / gain 0.8-1.2)."""
    if np.random.rand() < 0.5:
        img = np.ascontiguousarray(img[:, ::-1])
    if np.random.rand() < 0.5:
        ang = np.random.uniform(-7, 7)
        M = cv2.getRotationMatrix2D((IMG_SIZE/2, IMG_SIZE/2), ang, 1.0)
        img = cv2.warpAffine(img, M, (IMG_SIZE, IMG_SIZE), flags=cv2.INTER_LINEAR)
    if np.random.rand() < 0.5:
        img = np.clip(img * np.random.uniform(0.8, 1.2), 0, 1)
    return img.astype(np.float32)

def cyc_take(perm, i0, n):
    return np.take(perm, np.arange(i0, i0 + n) % len(perm))

def seg_batch(df, idx):
    xs, ms = [], []
    for i in idx:
        img, msk = load_tn3k_item(df.iloc[int(i)])
        img, msk = seg_aug(img, msk)
        xs.append(to_tensor(img)); ms.append(torch.from_numpy(msk[None]).float())
    return torch.stack(xs), torch.stack(ms)

def cls_batch(df, idx, train=True):
    xs, ys = [], []
    for i in idx:
        r = df.iloc[int(i)]
        img = letterbox_image(read_gray(r["image"]).astype(np.float32))
        if train:
            img = cls_aug(img)
        xs.append(to_tensor(img)); ys.append(np.float32(r["label"]))
    return torch.stack(xs), torch.from_numpy(np.array(ys, np.float32))

## 3. Joint trunk + classification heads
The trunk is the Phase-1 ResNet34-UNet (state-dict compatible with
`best_resnet34_seed42.pt`, loaded **strict**), exposing `features()` (e1..e4) and
`decode()` (seg logits). The classification descriptor levels are e3/e4 (768-d):

- **lesion head (JOINT-g):** training-time soft masked GAP with
  $w=\sigma(\text{seg logits})$ (area-downsampled per level; per-image
  normalisation); evaluation uses the locked hard R1p conventions (≥0.5 binarisation,
  nearest downsampling, <4 px empty-mask fallback to whole-image, flagged);
- **image head (JOINT-img):** plain per-image GAP over e3/e4.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = conv_block(cin + cskip, cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class JointUNet(nn.Module):
    """Phase-1 ResNet34-UNet trunk (state-dict compatible), fine-tuned unfrozen."""
    def __init__(self):
        super().__init__()
        import torchvision.models as tvm
        bb = tvm.resnet34(weights=None)
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu)
        self.pool = bb.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.up4, self.up3 = Up(512, 256, 256), Up(256, 128, 128)
        self.up2, self.up1 = Up(128, 64, 64), Up(64, 64, 64)
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            conv_block(64, 32), nn.Conv2d(32, 1, 1))
    def features(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(self.pool(e0)); e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3)
        return {"e0": e0, "e1": e1, "e2": e2, "e3": e3, "e4": e4}
    def decode(self, f):
        d = self.up1(self.up2(self.up3(self.up4(f["e4"], f["e3"]), f["e2"]), f["e1"]), f["e0"])
        return self.head(d)
    def forward(self, x):
        f = self.features(x)
        return self.decode(f), f

def build_trunk():
    trunk = JointUNet().to(DEVICE)
    _ck = load_ckpt(ART["ckpt_phase1"])
    trunk.load_state_dict(_ck["model"], strict=True)
    print(f"  trunk initialised from Phase 1 (which={_ck.get('which')}, "
          f"val Dice={_ck.get('val_dice'):.4f}) - UNFROZEN")
    return trunk

def build_head():
    return nn.Linear(768, 1).to(DEVICE)   # (e3 256 + e4 512) = 768-d descriptor -> 1 logit

def masked_gap(feat, msk01):
    """Hard per-image masked GAP (locked Phase-3 convention; nearest downsample)."""
    h, w = feat.shape[-2:]
    m = F.interpolate(msk01, size=(h, w), mode="nearest")
    den = m.sum(dim=(2, 3)).clamp(min=1.0)            # (B,1) per-image masked area
    return (feat * m).sum(dim=(2, 3)) / den           # (B, C) per-image descriptor

def soft_masked_gap(feat, w384):
    """Training-time soft masked GAP: w384 = sigmoid(seg logits), area-downsampled."""
    h, w = feat.shape[-2:]
    m = F.interpolate(w384, size=(h, w), mode="area")
    den = m.sum(dim=(2, 3)).clamp(min=1.0)            # (B,1) per-image
    return (feat * m).sum(dim=(2, 3)) / den           # (B, C)

def descriptor(feats, pool_mode, w384=None):
    if pool_mode == "image":
        return torch.cat([feats[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1)
    if w384 is None:
        raise ValueError("lesion mode requires mask weights")
    return torch.cat([soft_masked_gap(feats[nm], w384) for nm in FEAT_SCALES], 1)

def tversky_loss(logits, mask, alpha=TV_ALPHA, beta=TV_BETA, eps=1.0):
    p = torch.sigmoid(logits)
    tp = (p * mask).sum(dim=(2, 3))
    fp = (p * (1 - mask)).sum(dim=(2, 3))
    fn = ((1 - p) * mask).sum(dim=(2, 3))
    tv = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return (1 - tv).mean()

_X = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
_T = build_trunk().eval()
with torch.no_grad():
    _L, _F = _T(_X)
print("trunk shapes: logits", tuple(_L.shape), "| e3", tuple(_F["e3"].shape),
      "| e4", tuple(_F["e4"].shape), "| descriptor d =", (256 + 512))
_H = build_head(); print("head:", _H)
del _T, _H, _X, _L, _F

## 4. Joint training — two-phase v2 protocol, autologous arms

**Phase W (warmup).** Trunk frozen; classification head only (AdamW lr 1e-4 /
wd 1e-4, cosine T_max=25, 25 epochs, TN5000 train batches). This places the head
near its conditional minimum BEFORE unfreezing, so fresh-head gradients cannot
hijack the shared encoder (the v1 failure mechanism). A trunk-frozen tn3k-val
Dice is printed once at warmup end as a trunk-integrity probe (must match the
Phase-1 initialisation).

**Phase J (joint).** Per paired step: one TN3K batch (BCE + Tversky through the
seg decoder) and one TN5000 batch (BCE + pos_weight through the arm's
classification head) share one backward pass
($\mathcal{L} = \lambda_{seg}\mathcal{L}_{seg} + \lambda_{cls}\mathcal{L}_{cls}$,
λ=1 each). AdamW **lr 1e-5** / wd 1e-4 over trunk+head, cosine T_max=15,
≤15 epochs, patience 4, seed 42, AMP. Checkpoint selection maximises TN5000-val
AUROC **subject to tn3k-val Dice ≥ 0.8066** (`SEG_SELECT_GUARD`; if no epoch is
feasible, the best-Dice state is saved for diagnosis and §5 will fail loudly —
the honest negative path). **`set_seed(42)` is the first statement inside
`train_arm`** — the two arms see identical batch/augmentation streams and differ
ONLY in the classification pooling region.

In [ ]:
ytr = SPLITS["train"].label.to_numpy(np.float32)
POS_W = float((len(ytr) - ytr.sum()) / max(ytr.sum(), 1))
print(f"TN5000 train pos_weight = {POS_W:.4f}")

@torch.no_grad()
def eval_cls_probs(trunk, head, pool_mode, df, bs=BATCH):
    """Hard-eval classification probs (locked R1p conventions for lesion mode)."""
    trunk.eval(); head.eval(); ps, empty = [], 0
    for i0 in range(0, len(df), bs):
        x, _ = cls_batch(df, np.arange(i0, min(i0 + bs, len(df))), train=False)
        x = x.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits_seg, feats = trunk(x)
            prob_seg = torch.sigmoid(logits_seg.float())
        if pool_mode == "lesion":
            pm = (prob_seg >= PRED_THRESH).float()
            for bi in range(len(x)):
                if pm[bi].sum() < 4:                        # empty-mask fallback
                    pm[bi] = 1.0; empty += 1
            desc = torch.cat([masked_gap(feats[nm], pm) for nm in FEAT_SCALES], 1)
        else:
            desc = torch.cat([feats[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1)
        with torch.no_grad():
            ps.append(torch.sigmoid(head(desc.float()).squeeze(1)).float().cpu())
    return torch.cat(ps).numpy().ravel(), empty / max(1, len(df))

@torch.no_grad()
def eval_seg_dice(trunk, df, thr=0.5, bs=BATCH):
    trunk.eval(); d = []
    for i0 in range(0, len(df), bs):
        idx = np.arange(i0, min(i0 + bs, len(df)))
        xs, gts = [], []
        for i in idx:
            img, msk = load_tn3k_item(df.iloc[int(i)])
            xs.append(to_tensor(img)); gts.append(msk)
        x = torch.stack(xs).to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits_seg, _ = trunk(x)
        pm = (torch.sigmoid(logits_seg.float()) >= thr).float().cpu().numpy()
        for bi in range(len(xs)):
            d.append(dice_np(pm[bi, 0], gts[bi]))
    return float(np.mean(d))

def train_arm(pool_mode):
    set_seed(SEED)                                                 # identical streams, autologous arms
    assert pool_mode in ("lesion", "image")
    trunk = build_trunk(); head = build_head()
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_W], device=DEVICE))
    ns, nc = len(tn3k_train), len(SPLITS["train"])
    steps = int(np.ceil(max(ns, nc) / BATCH))
    hist = []

    # -------------------- phase W: trunk-frozen head warmup (v2) -----------------
    for p in trunk.parameters(): p.requires_grad_(False)
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_WARM)
    nwc = int(np.ceil(nc / BATCH))
    best_w, best_hw = -1.0, None
    t0 = time.time()
    for ep in range(1, EPOCHS_WARM + 1):
        trunk.eval(); head.train()
        perm_c = np.random.permutation(nc)
        lc = 0.0
        for st in range(nwc):
            xc, yc = cls_batch(SPLITS["train"], cyc_take(perm_c, st * BATCH, BATCH), train=True)
            xc = xc.to(DEVICE, non_blocking=True); yc = yc.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                with torch.no_grad():
                    logits_c, feats_c = trunk(xc)
                    if pool_mode == "lesion":
                        w384 = torch.sigmoid(logits_c)
                        desc = descriptor(feats_c, "lesion", w384)
                    else:
                        desc = descriptor(feats_c, "image")
                loss_cls = bce(head(desc.float()).squeeze(1), yc)
            opt.zero_grad(set_to_none=True); loss_cls.backward(); opt.step()
            lc += float(loss_cls)
        sched.step()
        pv, _e = eval_cls_probs(trunk, head, pool_mode, SPLITS["val"])
        va = roc_auc_score(SPLITS["val"].label.to_numpy(), pv)
        star = ""
        if va > best_w + 1e-4:
            best_w = va
            best_hw = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            star = "  *"
        hist.append({"phase": "W", "epoch": ep, "loss_seg": float("nan"),
                     "loss_cls": lc / nwc, "val_auroc": float(va),
                     "tn3k_val_dice": float("nan"), "seg_feasible": True})
        print(f"  [{pool_mode}|W] ep {ep:02d} | cls {lc/nwc:.4f} | val AUROC {va:.4f}{star} "
              f"({(time.time()-t0)/60:.1f} min)", flush=True)
    head.load_state_dict(best_hw)
    d_frozen = eval_seg_dice(trunk, tn3k_val, thr=0.5)
    print(f"  [{pool_mode}|W] warmup done: best head val AUROC {best_w:.4f}; "
          f"trunk-frozen tn3k-val Dice {d_frozen:.4f} (trunk-integrity probe)", flush=True)

    # ------------- phase J: low-LR joint unfreeze, seg-guarded selection (v2) ---
    for p in trunk.parameters(): p.requires_grad_(True)
    params = list(trunk.parameters()) + list(head.parameters())
    opt = torch.optim.AdamW(params, lr=LR_JOINT, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    best, wait, best_state = -1.0, 0, None
    best_dv, best_state_dv = -1.0, None
    n_feasible = 0
    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        trunk.train(); head.train()
        perm_s = np.random.permutation(ns); perm_c = np.random.permutation(nc)
        ls = lc = 0.0
        for st in range(steps):
            xs, ms = seg_batch(tn3k_train, cyc_take(perm_s, st * BATCH, BATCH))
            xc, yc = cls_batch(SPLITS["train"], cyc_take(perm_c, st * BATCH, BATCH), train=True)
            xs = xs.to(DEVICE, non_blocking=True); ms = ms.to(DEVICE, non_blocking=True)
            xc = xc.to(DEVICE, non_blocking=True); yc = yc.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                logits_s, _ = trunk(xs)
                loss_seg = F.binary_cross_entropy_with_logits(logits_s, ms) + tversky_loss(logits_s, ms)
                logits_c, feats_c = trunk(xc)
                if pool_mode == "lesion":
                    w384 = torch.sigmoid(logits_c)
                    desc = descriptor(feats_c, "lesion", w384)
                else:
                    desc = descriptor(feats_c, "image")
                loss_cls = bce(head(desc).squeeze(1), yc)
                loss = LAMBDA_SEG * loss_seg + LAMBDA_CLS * loss_cls
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            ls += float(loss_seg); lc += float(loss_cls)
        sched.step()
        pv, _e = eval_cls_probs(trunk, head, pool_mode, SPLITS["val"])
        va = roc_auc_score(SPLITS["val"].label.to_numpy(), pv)
        dv = eval_seg_dice(trunk, tn3k_val, thr=0.5)
        feasible = dv >= SEG_SELECT_GUARD
        n_feasible += int(feasible)
        if dv > best_dv + 1e-4:
            best_dv = dv
            best_state_dv = {"trunk": {k: v.detach().cpu().clone() for k, v in trunk.state_dict().items()},
                             "head":  {k: v.detach().cpu().clone() for k, v in head.state_dict().items()},
                             "pool_mode": pool_mode, "val_auroc": float(va), "epoch": ep,
                             "tn3k_val_dice": float(dv)}
        star = ""
        if feasible and va > best + 1e-4:
            best, wait = va, 0
            best_state = {"trunk": {k: v.detach().cpu().clone() for k, v in trunk.state_dict().items()},
                          "head":  {k: v.detach().cpu().clone() for k, v in head.state_dict().items()},
                          "pool_mode": pool_mode, "val_auroc": float(va), "epoch": ep,
                          "tn3k_val_dice": float(dv)}
            star = "  *"
        else:
            wait += 1
        hist.append({"phase": "J", "epoch": ep, "loss_seg": ls / steps, "loss_cls": lc / steps,
                     "val_auroc": float(va), "tn3k_val_dice": dv, "seg_feasible": bool(feasible)})
        print(f"  [{pool_mode}|J] ep {ep:02d} | seg {ls/steps:.4f} cls {lc/steps:.4f} | "
              f"val AUROC {va:.4f} | tn3k-val Dice {dv:.4f}{'' if feasible else ' [seg-infeasible]'}{star} "
              f"({(time.time()-t0)/60:.1f} min)", flush=True)
        if wait >= PATIENCE:
            print(f"  [{pool_mode}|J] early stop at ep {ep} (best guarded {best:.4f})"); break
    seg_feasible_any = best_state is not None
    if not seg_feasible_any:
        print(f"  [{pool_mode}|J] WARNING: no joint epoch met the seg guard "
              f"({SEG_SELECT_GUARD:.4f}); saving the best-Dice state for diagnosis. "
              f"The sec-5 gate is EXPECTED to fail - honest negative path.")
        best_state = best_state_dv
    best_state["seg_feasible_any"] = bool(seg_feasible_any)
    best_state["n_joint_feasible"] = int(n_feasible)
    trunk.load_state_dict(best_state["trunk"]); head.load_state_dict(best_state["head"])
    torch.save(best_state, OUTPUT_ROOT / f"joint_{'g' if pool_mode=='lesion' else 'img'}_trunk.pt")
    return {"trunk": trunk, "head": head, "pool_mode": pool_mode,
            "best_val_auroc": float(best_state["val_auroc"]),
            "tn3k_val_dice_saved": float(best_state["tn3k_val_dice"]),
            "seg_feasible_any": bool(seg_feasible_any),
            "n_joint_feasible": int(n_feasible), "history": hist}

ARMS = {}
for mode in ["lesion", "image"]:
    print(f"=== training JOINT-{'g' if mode=='lesion' else 'img'} arm (v2: warmup W + guard-gated joint J) ===")
    ARMS[mode] = train_arm(mode)
assert (ARMS["lesion"]["best_val_auroc"] is not None) and (ARMS["image"]["best_val_auroc"] is not None)
msg = ("arms done | JOINT-g val AUROC %.4f (val Dice %.4f, feasible %s) | "
       "JOINT-img val AUROC %.4f (val Dice %.4f, feasible %s)"
       % (ARMS["lesion"]["best_val_auroc"], ARMS["lesion"]["tn3k_val_dice_saved"],
          ARMS["lesion"]["seg_feasible_any"],
          ARMS["image"]["best_val_auroc"], ARMS["image"]["tn3k_val_dice_saved"],
          ARMS["image"]["seg_feasible_any"]))
print(msg)

## 5. Gates — segmentation non-collapse + frozen-row integrity
1. **Seg gate (hard):** each joint trunk must reach TN3K-test Dice ≥ locked 0.8466 − 0.05
   at the locked 0.60 threshold (no TTA, no refitting); TN3K-test is the only hard gate.
2. **Frozen-row gates (hard):** locked ERM TN5000-test probs must re-score ±0.002 of
   0.9401; locked R1p probs ±0.002 of 0.8902 — else the family-G comparisons are void.

In [ ]:
print("segmentation non-collapse gate (TN3K test, thr=0.60):")
SEGQC = {}
for mode in ["lesion", "image"]:
    dte = eval_seg_dice(ARMS[mode]["trunk"], tn3k_test, thr=SEG_EVAL_THRESH)
    SEGQC[mode] = {"tn3k_test_dice": dte}
    ok = dte >= SUPERVISED_REF["TN3K-test"] - SEG_GATE_TOL
    print(f"  JOINT-{'g' if mode=='lesion' else 'img'} trunk: {dte:.4f} "
          f"(locked supervised TTA ref {SUPERVISED_REF['TN3K-test']:.4f}; "
          f"gate >= {SUPERVISED_REF['TN3K-test'] - SEG_GATE_TOL:.4f}) {'OK' if ok else 'FAIL'}")
    assert ok, f"seg non-collapse gate failed for {mode} trunk"

# ---------- frozen locked rows (never retrained here) ----------
ERMJ  = json.loads(ART["erm_probs"].read_text())            # {"val_auroc", "probs": {train,val,test}}
LINORIG = json.loads(ART["lin_heads"].read_text())          # Phase-3 locked linear-head probs
_zv = np.load(ART["tn5k_fval"], allow_pickle=True)
_zt = np.load(ART["tn5k_ftest"], allow_pickle=True)
y_val = _zv["labels"].astype(int).ravel()
y_test = _zt["labels"].astype(int).ravel()
assert len(y_val) == 476 and len(y_test) == 950, "feature-bank labels do not match the gated manifest"

def locked_probs(src, split):
    return np.asarray(src, float).ravel()

P_VAL = {"ERM": locked_probs(ERMJ["probs"]["val"], "val"),
         "R1p": locked_probs(LINORIG["R1p_lesion"]["probs_val"], "val"),
         "R0":  locked_probs(LINORIG["R0_image"]["probs_val"], "val")}
P_TEST = {"ERM": locked_probs(ERMJ["probs"]["test"], "test"),
          "R1p": locked_probs(LINORIG["R1p_lesion"]["probs_test"], "test"),
          "R0":  locked_probs(LINORIG["R0_image"]["probs_test"], "test")}

erm_chk = roc_auc_score(y_test, P_TEST["ERM"])
r1p_chk = roc_auc_score(y_test, P_TEST["R1p"])
print(f"ERM sanity gate:    {erm_chk:.4f} vs locked {ERM_REF_AUROC} (tol {GATE_TOL_AUROC})")
print(f"R1p integrity gate: {r1p_chk:.4f} vs locked {R1P_REF_AUROC} (tol {GATE_TOL_AUROC})")
assert abs(erm_chk - ERM_REF_AUROC) <= GATE_TOL_AUROC, "ERM sanity gate failed"
assert abs(r1p_chk - R1P_REF_AUROC) <= GATE_TOL_AUROC, "R1p cache integrity gate failed"

LINR = json.loads(ART["lin_retr"].read_text())              # 3c retrained head params (for external R1p)
def lin_apply(rep, X):
    h = LINR[rep]["head"]
    w = np.array(h["w"]); b = np.array(h["b"]); mu = np.array(h["mu"]); sd = np.array(h["sd"])
    z = ((X.astype(np.float64) - mu) / sd) @ w.T + b
    return (1 / (1 + np.exp(-z.ravel()))).astype(float)
P_VAL["R1p_ext"]  = np.asarray(LINR["R1p_lesion"]["probs"]["val"], float).ravel()   # ext threshold source
print("frozen rows loaded: ERM + R0/R1p (locked) | R1p_ext head params (3c retrain) ready")

## 6. Metrics (torch-free) — locked conventions
Thresholds and temperature from **TN5000-val only**; raw-raw operating points
(threshold on raw val probs applied to raw eval probs), calibration metrics on
temperature-calibrated probabilities; never mixed. AUROC/AUPRC convention-invariant.
CIs: case bootstrap (2000 main, 5000 family-G paired, 2000 family-H paired);
two-sided sign-tail p; McNemar with continuity correction on discordant malignant
cases. Selective prediction: coverage vs malignant-FNR at the frozen raw threshold,
ranked by calibrated confidence (margin ranking is identical for binary entropy).

In [ ]:
def logit(p):
    p = np.clip(np.asarray(p, np.float64).ravel(), 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def fit_temperature(y, p):
    z, yv = logit(p), np.asarray(y, np.float64).ravel()
    grid = np.exp(np.linspace(np.log(0.05), np.log(20.0), 200))
    def nll(T):
        q = 1 / (1 + np.exp(-z / T))
        return -np.mean(yv * np.log(q + 1e-12) + (1 - yv) * np.log(1 - q + 1e-12))
    return float(grid[int(np.argmin([nll(T) for T in grid]))])

def ece(y, p, bins=15):
    y = np.asarray(y, np.float64).ravel(); p = np.asarray(p, np.float64).ravel()
    edges = np.linspace(0, 1, bins + 1); out = 0.0
    for i in range(bins):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < bins - 1 else p <= edges[i + 1])
        if m.sum():
            out += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(out)

def spec90_threshold(y_val, p_val):
    fpr, tpr, thr = roc_curve(np.asarray(y_val).ravel(), np.asarray(p_val).ravel())
    spec = 1 - fpr
    m = spec >= 0.90
    if m.any():
        i = np.where(m)[0][np.argmax(tpr[m])]
    else:
        i = int(np.argmax(spec))
    return float(thr[i]), float(tpr[i]), float(spec[i])

def brier(y, p):
    return float(np.mean((np.asarray(p, np.float64).ravel() - np.asarray(y, np.float64).ravel()) ** 2))

def metric_set(y, p, thr):
    y = np.asarray(y).ravel(); p = np.asarray(p).ravel()
    return {"auroc": roc_auc_score(y, p), "auprc": average_precision_score(y, p),
            "sens@spec90": float((p[y == 1] >= thr).mean()), "brier": brier(y, p),
            "ece": ece(y, p)}

def boot_ci(y, p, thr, n=N_BOOT_MAIN, seed=SEED):
    y, p = np.asarray(y).ravel(), np.asarray(p).ravel()
    rng = np.random.RandomState(seed); N = len(y); acc = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2:
            continue
        acc.append([roc_auc_score(y[idx], p[idx]), average_precision_score(y[idx], p[idx]),
                    (p[idx][y[idx] == 1] >= thr).mean()])
    lo, hi = np.percentile(acc, 2.5, 0), np.percentile(acc, 97.5, 0)
    return lo, hi

def paired_boot_auroc(y, pa, pb, n=N_BOOT_G, seed=SEED):
    y = np.asarray(y).ravel(); rng = np.random.RandomState(seed); N = len(y); d = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2:
            continue
        d.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.array(d)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), min(p, 1.0), len(d)

def paired_boot_sens(y, pa, ta, pb, tb, n=N_BOOT_G, seed=SEED):
    y = np.asarray(y).ravel(); pos = y == 1
    rng = np.random.RandomState(seed)
    sa = (pa[pos] >= ta).astype(float); sb = (pb[pos] >= tb).astype(float)
    M = len(sa); d = []
    for _ in range(n):
        idx = rng.randint(0, M, M)
        d.append(sa[idx].mean() - sb[idx].mean())
    d = np.array(d)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), min(p, 1.0), len(d)

def mcnemar_chi2(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    if b + c == 0:
        return None, b, c
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return chi2, b, c

def coverage_curve(y, p_raw, thr, conf, grid):
    """Retained-set FNR and sensitivity at the frozen raw threshold, ranking cases by
    descending calibrated confidence conf = |p_cal - 0.5| (binary-margin ranking)."""
    y = np.asarray(y).ravel(); p_raw = np.asarray(p_raw).ravel(); conf = np.asarray(conf).ravel()
    order = np.argsort(-conf, kind="stable")
    y, p_raw = y[order], p_raw[order]
    pos_tot = max(1, int((y == 1).sum()))
    rows = []
    for c in grid:
        k = max(1, int(round(c * len(y))))
        yr, pr = y[:k], p_raw[:k]
        fn = int(((yr == 1) & (pr < thr)).sum())
        rows.append({"coverage": float(c), "n_retained": k,
                     "malignant_FNR": fn / max(1, int((yr == 1).sum())),
                     "sens@spec90": float((pr[yr == 1] >= thr).mean()) if (yr == 1).sum() else np.nan,
                     "malignant_share_retained": float((yr == 1).sum() / pos_tot)})
    return rows

## 7. In-domain evaluation — TN5000 val (thresholds/temperature) and test
Both arms under the locked hard-eval conventions. Locked Phase-3 rows (ERM, R0, R1p)
shown for context from cached probabilities — never retrained.

In [ ]:
P_VAL["JOINTg"]  = eval_cls_probs(ARMS["lesion"]["trunk"], ARMS["lesion"]["head"],
                                    "lesion", SPLITS["val"])[0].ravel()
P_TEST["JOINTg"], emp_gte = eval_cls_probs(ARMS["lesion"]["trunk"], ARMS["lesion"]["head"],
                                           "lesion", SPLITS["test"])
P_VAL["JOINTimg"]  = eval_cls_probs(ARMS["image"]["trunk"], ARMS["image"]["head"],
                                    "image", SPLITS["val"])[0].ravel()
P_TEST["JOINTimg"], emp_ite = eval_cls_probs(ARMS["image"]["trunk"], ARMS["image"]["head"],
                                             "image", SPLITS["test"])
P_TEST["JOINTg"] = P_TEST["JOINTg"].ravel(); P_TEST["JOINTimg"] = P_TEST["JOINTimg"].ravel()
print(f"empty-mask fallback (JOINT-g, test): {emp_gte*100:.2f}% "
      f"(JOINT-img has no masks: n/a, echo {emp_ite*0:.0f}%)")

THR, TEMP = {}, {}
for m in ["JOINTg", "JOINTimg", "ERM", "R1p", "R0"]:
    thr, sv, sp = spec90_threshold(y_val, P_VAL[m])          # raw val probs -> frozen threshold
    THR[m] = thr
    TEMP[m] = fit_temperature(y_val, P_VAL[m])               # val logits -> frozen temperature

rows = []
for m in ["JOINTg", "JOINTimg", "ERM", "R1p", "R0"]:
    pt = P_TEST[m].ravel()
    ptc = 1 / (1 + np.exp(-logit(pt) / TEMP[m]))
    ms = metric_set(y_test, pt, THR[m])                      # raw-raw: raw probs at raw threshold
    ms["brier"] = brier(y_test, ptc)                         # calibration on calibrated probs
    ms["ece"] = ece(y_test, pt); ms["ece_post"] = ece(y_test, ptc)
    lo, hi = boot_ci(y_test, pt, THR[m])
    rows.append({"model": m, "auroc": round(ms["auroc"], 4),
                 "auroc_CI": f"({lo[0]:.4f}-{hi[0]:.4f})",
                 "auprc": round(ms["auprc"], 4), "sens@spec90": round(ms["sens@spec90"], 4),
                 "brier": round(ms["brier"], 4), "ece_pre": round(ms["ece"], 4),
                 "ece_post": round(ms["ece_post"], 4), "T_val": round(TEMP[m], 3)})
MAIN_IN = pd.DataFrame(rows)
MAIN_IN.to_csv(OUTPUT_ROOT / "phase4_indomain_table.csv", index=False)
print("TN5000 test (n=%d, malignant=%d) — thresholds/temperature from val (raw-raw):"
      % (len(y_test), int(y_test.sum())))
display(MAIN_IN)

json.dump({"val": {m: P_VAL[m].ravel().tolist() for m in ["JOINTg", "JOINTimg"]},
           "test": {m: P_TEST[m].ravel().tolist() for m in ["JOINTg", "JOINTimg"]},
           "thr": THR, "temp": TEMP},
          open(OUTPUT_ROOT / "phase4_joint_probs_tn5000.json", "w"), indent=1)
print("saved phase4_joint_probs_tn5000.json")

## 8. Family G — pre-registered in-domain paired tests (Bonferroni α = 0.05/4 = 0.0125)
G1 `JOINTg − ERM` AUROC — *is the Phase-3b T2 gap (+0.0498 ERM over frozen R1p) closed?* ·
G2 `JOINTg − R1p` AUROC — *what does jointing add over frozen grounding?* ·
G3 `JOINTg − JOINTimg` AUROC — *grounding vs mere multi-tasking, autologous arms* ·
G4 `sens@spec90 JOINTg − ERM` (+ McNemar, raw-raw).

In [ ]:
rows = []
def rec(test, kind, a, b, mean, lo, hi, p, n, alpha, extra=""):
    rows.append({"test": test, "metric": kind, "A": a, "B": b,
                 "delta(A-B)": round(mean, 4), "CI95": f"({lo:.4f}..{hi:.4f})",
                 "p_two_sided": round(p, 5),
                 "sig_bonf": "YES" if p < alpha else "no", "n_boot": n, "note": extra})

mean, lo, hi, p, n = paired_boot_auroc(y_test, P_TEST["JOINTg"], P_TEST["ERM"], N_BOOT_G)
rec("G1", "AUROC", "JOINTg", "ERM", mean, lo, hi, p, n, ALPHA_G, "is the T2 fine-tune gap closed?")
mean, lo, hi, p, n = paired_boot_auroc(y_test, P_TEST["JOINTg"], P_TEST["R1p"], N_BOOT_G)
rec("G2", "AUROC", "JOINTg", "R1p", mean, lo, hi, p, n, ALPHA_G, "joint vs frozen grounding")
mean, lo, hi, p, n = paired_boot_auroc(y_test, P_TEST["JOINTg"], P_TEST["JOINTimg"], N_BOOT_G)
rec("G3", "AUROC", "JOINTg", "JOINTimg", mean, lo, hi, p, n, ALPHA_G, "grounding vs multi-tasking")
mean, lo, hi, p, n = paired_boot_sens(y_test, P_TEST["JOINTg"], THR["JOINTg"],
                                      P_TEST["ERM"], THR["ERM"], N_BOOT_G)
rec("G4", "sens@spec90", "JOINTg", "ERM", mean, lo, hi, p, n, ALPHA_G, "operating point")

chi2, b_, c_ = mcnemar_chi2((P_TEST["JOINTg"][y_test == 1] >= THR["JOINTg"]),
                            (P_TEST["ERM"][y_test == 1] >= THR["ERM"]))
print(f"McNemar (G4, test positives): JOINTg right / ERM wrong = {b_}, "
      f"ERM right / JOINTg wrong = {c_}, chi2 = "
      f"{chi2 if chi2 is None else round(chi2, 2)} (3.84 ~ p<0.05, 6.63 ~ p<0.01)")

FAMG = pd.DataFrame(rows)
FAMG.to_csv(OUTPUT_ROOT / "phase4_familyG.csv", index=False)
print(f"Bonferroni alpha = {ALPHA_G:.4f} across 4 in-domain tests")
display(FAMG)

## 9. External battery — TNUI test/val/train + DDTI on the frozen conventions
Row order and labels come **only** from the locked Phase-3c/3d/3e feature banks (hard
alignment assert against the image folders; no label re-parsing, no manifest edits).
JOINT arms score through their own trunks (hard-eval lesion descriptors; segmentation
QC vs cohort masks reported alongside). ERM external scores are recomputed from the
archived checkpoint (sanity-gated above); R1p external scores come from the 3c-retrained
head applied to the locked cached descriptors (`lin_apply`). Thresholds per model from
its own TN5000-val raw probs; external metrics raw-raw.

In [ ]:
EXT_DEF = [
    ("TNUI-test",  ART["bank_tnui_test"],  PATHS["tnui"]/"test"/"images",  PATHS["tnui"]/"test"/"masks",  "R1g_gtmask"),
    ("TNUI-val",   ART["bank_tnui_val"],   PATHS["tnui"]/"val"/"images",   PATHS["tnui"]/"val"/"masks",   "R1g_gtmask"),
    ("TNUI-train", ART["bank_tnui_train"], PATHS["tnui"]/"train"/"images", PATHS["tnui"]/"train"/"masks", None),
    ("DDTI",       ART["bank_ddti"],       DDTI_ROOT/"1_or_data"/"image",  DDTI_ROOT/"1_or_data"/"mask",  None),
]

@torch.no_grad()
def eval_erm_probs(model, df, bs=BATCH):
    model.eval(); ps = []
    for i0 in range(0, len(df), bs):
        x, _ = cls_batch(df, np.arange(i0, min(i0 + bs, len(df))), train=False)
        x = x.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            ps.append(torch.sigmoid(model(x)).squeeze(1).float().cpu())
    return torch.cat(ps).numpy().ravel()

import torchvision.models as tvm
erm = tvm.resnet34(weights=None); erm.fc = nn.Linear(erm.fc.in_features, 1)
erm.load_state_dict(load_ckpt(ART["erm_ckpt"])["model"]); erm = erm.to(DEVICE).eval()

@torch.no_grad()
def run_joint_bank(arm, df, gtm=None, bs=BATCH):
    """Return per-case probs + joint segmentation QC for one external cohort."""
    trunk, head = arm["trunk"], arm["head"]
    trunk.eval(); head.eval()
    ps, dice, empty, area = [], [], 0, []
    for i0 in range(0, len(df), bs):
        x, _ = cls_batch(df, np.arange(i0, min(i0 + bs, len(df))), train=False)
        x = x.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits_seg, feats = trunk(x)
        prob_seg = torch.sigmoid(logits_seg.float())
        pm = (prob_seg >= PRED_THRESH).float()
        pm_raw = pm.clone()
        for bi in range(len(x)):
            if pm[bi].sum() < 4:
                pm[bi] = 1.0; empty += 1
        if arm["pool_mode"] == "lesion":
            desc = torch.cat([masked_gap(feats[nm], pm) for nm in FEAT_SCALES], 1)
        else:
            desc = torch.cat([feats[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1)
        ps.append(torch.sigmoid(head(desc.float()).squeeze(1)).float().cpu())
        area.extend(pm_raw.mean(dim=(1, 2, 3)).tolist())
        if gtm is not None:
            pmn = pm_raw.cpu().numpy()
            for bi in range(len(x)):
                dice.append(dice_np(pmn[bi, 0], gtm[i0 + bi]))
    out = {"probs": torch.cat(ps).numpy().ravel(),
           "dice_mean": float(np.mean(dice)) if dice else np.nan,
           "empty_rate": empty / max(1, len(df)), "area_mean": float(np.mean(area))}
    return out

EXT = {}
t0_all = time.time()
for name, bank_path, img_dir, msk_dir, _ in EXT_DEF:
    zb = np.load(bank_path, allow_pickle=True)
    bank = {k: zb[k] for k in zb.files}
    ids_ext = [str(i) for i in bank["ids"].tolist()]
    imgs = {p.stem: p for p in image_files(img_dir)}
    found = [i for i in ids_ext if i in imgs]
    assert len(found) == len(ids_ext), \
        f"{name}: image-folder alignment failed ({len(found)}/{len(ids_ext)} ids found) - " \
        f"ids must equal cache ids exactly; no manifest re-derivation permitted"
    df = pd.DataFrame({"id": ids_ext, "image": [str(imgs[i]) for i in ids_ext],
                       "label": bank["labels"].astype(int)})
    assert len(df) == len(bank["labels"]), f"{name}: bank/df row misalignment"
    gtm = None
    if msk_dir is not None and Path(msk_dir).exists():
        msks = {p.stem: p for p in image_files(msk_dir)}
        if all(i in msks for i in ids_ext):
            gtm = [letterbox_mask(read_gray(str(msks[i]))) for i in ids_ext]
    ext = {"y": df.label.to_numpy(), "df": df}
    for key, mode in [("JOINTg", "lesion"), ("JOINTimg", "image")]:
        r = run_joint_bank(ARMS[mode], df, gtm)
        ext[key] = r["probs"]
        ext[key + "_qc"] = {k: r[k] for k in ("dice_mean", "empty_rate", "area_mean")}
    ext["ERM"] = eval_erm_probs(erm, df)
    ext["R1p"] = lin_apply("R1p_lesion", bank["R1p_lesion"])
    EXT[name] = ext
    print(f"{name:11s} n={len(df):4d} malignant={int(df.label.sum()):4d} | "
          f"JOINT-g seg Dice {ext['JOINTg_qc']['dice_mean']:.4f} | "
          f"empty {ext['JOINTg_qc']['empty_rate']*100:.1f}% | "
          f"({(time.time()-t0_all)/60:.1f} min elapsed)", flush=True)

THR_EXT = {"JOINTg": THR["JOINTg"], "JOINTimg": THR["JOINTimg"],
           "ERM": spec90_threshold(y_val, P_VAL["ERM"])[0],
           "R1p": spec90_threshold(y_val, P_VAL["R1p_ext"])[0]}
rows = []
for name in EXT:
    ext = EXT[name]
    for m in ["JOINTg", "JOINTimg", "ERM", "R1p"]:
        pr = ext[m].ravel()
        rows.append({"cohort": name, "n": len(ext["y"]), "malignant": int(ext["y"].sum()),
                     "model": m, "auroc": round(roc_auc_score(ext["y"], pr), 4),
                     "sens@spec90": round(float((pr[ext["y"] == 1] >= THR_EXT[m]).mean()), 4)})
EXT_TAB = pd.DataFrame(rows)
EXT_TAB.to_csv(OUTPUT_ROOT / "phase4_external_table.csv", index=False)
print("\nExternal battery (thresholds from each model's TN5000-val raw probs; raw-raw):")
display(EXT_TAB.pivot_table(index=["cohort", "n", "malignant"], columns="model",
                            values=["auroc", "sens@spec90"]).round(4))
json.dump({name: {m: EXT[name][m].ravel().tolist() for m in ["JOINTg", "JOINTimg", "ERM", "R1p"]}
           for name in EXT}, open(OUTPUT_ROOT / "phase4_external_probs.json", "w"))
print("saved phase4_external_probs.json")

## 10. Family H — pre-registered pooled TNUI test+val (the single locked pool)
Bonferroni α = 0.05/3 = 0.0167; 2000 paired resamples.
H1 `sens@spec90 JOINTg − ERM` (**primary**, operating point) ·
H2 `sens@spec90 JOINTg − R1p` · H3 `AUROC JOINTg − ERM`.
TNUI-train and DDTI rows above remain descriptive per the locked pooling policy
(no new pooling, no inference beyond the pre-declared family).

In [ ]:
y_pool  = np.concatenate([EXT["TNUI-test"]["y"], EXT["TNUI-val"]["y"]])
P_POOL  = {m: np.concatenate([EXT["TNUI-test"][m].ravel(), EXT["TNUI-val"][m].ravel()])
           for m in ["JOINTg", "JOINTimg", "ERM", "R1p"]}
print(f"pooled TNUI test+val: n={len(y_pool)} (malignant={int(y_pool.sum())}, benign={int((1-y_pool).sum())})")

rows = []
def rech(test, kind, a, b, mean, lo, hi, p, n, extra=""):
    rows.append({"test": test, "metric": kind, "A": a, "B": b,
                 "delta(A-B)": round(mean, 4), "CI95": f"({lo:.4f}..{hi:.4f})",
                 "p_two_sided": round(p, 5),
                 "sig_bonf": "YES" if p < ALPHA_H else "no", "n_boot": n, "note": extra})

mean, lo, hi, p, n = paired_boot_sens(y_pool, P_POOL["JOINTg"], THR_EXT["JOINTg"],
                                      P_POOL["ERM"], THR_EXT["ERM"], N_BOOT_H)
rech("H1", "sens@spec90", "JOINTg", "ERM", mean, lo, hi, p, n, "PRIMARY external operating point")
mean, lo, hi, p, n = paired_boot_sens(y_pool, P_POOL["JOINTg"], THR_EXT["JOINTg"],
                                      P_POOL["R1p"], THR_EXT["R1p"], N_BOOT_H)
rech("H2", "sens@spec90", "JOINTg", "R1p", mean, lo, hi, p, n, "joint vs frozen grounding, external")
mean, lo, hi, p, n = paired_boot_auroc(y_pool, P_POOL["JOINTg"], P_POOL["ERM"], N_BOOT_H)
rech("H3", "AUROC", "JOINTg", "ERM", mean, lo, hi, p, n, "external rank (secondary)")

chi2, b_, c_ = mcnemar_chi2((P_POOL["JOINTg"][y_pool == 1] >= THR_EXT["JOINTg"]),
                            (P_POOL["ERM"][y_pool == 1] >= THR_EXT["ERM"]))
print(f"McNemar (H1, pooled positives): JOINTg right / ERM wrong = {b_}, "
      f"ERM right / JOINTg wrong = {c_}, chi2 = "
      f"{chi2 if chi2 is None else round(chi2, 2)}")

FAMH = pd.DataFrame(rows)
FAMH.to_csv(OUTPUT_ROOT / "phase4_familyH.csv", index=False)
print(f"Bonferroni alpha = {ALPHA_H:.4f} across 3 pooled tests")
display(FAMH)

print("Descriptive context (no inference; locked pooling policy):")
desc = EXT_TAB[(EXT_TAB.cohort.isin(["TNUI-train", "DDTI"])) &
               (EXT_TAB.model.isin(["JOINTg", "ERM", "R1p"]))]
display(desc)

## 11. Uncertainty and selective prediction (Gap-6 pre-registered component)
Referral ranking by calibrated confidence |p_cal − 0.5| of JOINT-g (temperature fitted
on val, frozen). Coverage sweep retains the most confident cases; malignant-FNR and
sensitivity are evaluated at the frozen raw spec≥0.90 threshold on the retained set.
Reported on TN5000-test and on the pooled TNUI cohort. Assistive mechanism, not an
autonomous rule.

In [ ]:
T_g = TEMP["JOINTg"]
def cal(p):
    return 1 / (1 + np.exp(-logit(p) / T_g))

GRID = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
SP = {}
for name, y, praw in [("TN5000-test", y_test, P_TEST["JOINTg"]),
                      ("TNUI pooled (test+val)", y_pool, P_POOL["JOINTg"])]:
    conf = np.abs(cal(praw) - 0.5)
    SP[name] = coverage_curve(y, praw, THR_EXT["JOINTg"] if "TNUI" in name else THR["JOINTg"],
                              conf, GRID)
    tbl = pd.DataFrame(SP[name])
    print(f"selective prediction — {name} (thr frozen from val, raw-raw):")
    display(tbl.round(4))
    pd.DataFrame(SP[name]).to_csv(
        OUTPUT_ROOT / f"phase4_selective_{'tn5000' if 'TN5000' in name else 'tnui'}.csv",
        index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (name, rows_) in zip(axes, SP.items()):
    cov = [r["coverage"] for r in rows_]; fnr = [r["malignant_FNR"] for r in rows_]
    sen = [r["sens@spec90"] for r in rows_]
    ax.plot([100 * c for c in cov], fnr, "o-", label="malignant FNR")
    ax.plot([100 * c for c in cov], sen, "s--", label="sens@spec90")
    ax.set_xlabel("coverage (%) — most confident retained first")
    ax.set_ylim(0, 1); ax.set_title(name); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle(f"JOINT-g selective prediction (T={T_g:.2f} from val) — coverage vs error at frozen operating point")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "phase4_selective.png", dpi=140)
plt.show()
print("Gap-6 note: margin ranking equals binary-entropy ranking;")
print("FNR should fall as coverage drops if confidence carries referral information.")

## 12. Run summary, artifacts, and what to paste back

**Send back (one by one):**
1. §4 both training logs (phase W warmup lines, then phase J per-epoch val AUROC +
   tn3k-val Dice with feasibility tags, best epochs, arms-done line);
2. §5 seg gate + frozen-row gate lines;
3. §7 in-domain table (JOINTg/JOINTimg + locked context rows);
4. §8 family G table + McNemar line;
5. §9 external pivot (4 cohorts × 4 models, auroc + sens@spec90) + seg-QC line per cohort;
6. §10 family H table + McNemar line;
7. §11 selective-prediction tables + figure description;
8. Total wall time.

**Design boundaries preserved:** arms differ only in pooling region (autologous streams);
frozen rows never retrained (gates enforced); thresholds/temperature TN5000-val only
(raw-raw); the single locked pool governs external inference; DDTI/TNUI-train
descriptive only; joint seg non-collapse hard-gated; checkpoint selection seg-guarded
on tn3k-VAL only (test untouched); protocol v2 two-phase amendment logged.

In [ ]:
run = {"phase": "4 - joint partial-supervision fine-tune",
       "seed": SEED, "device": str(DEVICE),
       "protocol_version": "v2 (two-phase: trunk-frozen warmup + low-LR joint; seg-guarded selection)",
       "protocol": {"EPOCHS": EPOCHS, "PATIENCE": PATIENCE, "LR": LR, "WD": WD,
                    "LR_JOINT": LR_JOINT, "EPOCHS_WARM": EPOCHS_WARM,
                    "SEG_SELECT_GUARD": SEG_SELECT_GUARD,
                    "LAMBDA_SEG": LAMBDA_SEG, "LAMBDA_CLS": LAMBDA_CLS,
                    "PRED_THRESH": PRED_THRESH, "SEG_EVAL_THRESH": SEG_EVAL_THRESH,
                    "FEAT_SCALES": FEAT_SCALES, "BATCH": BATCH,
                    "ALPHA_G": ALPHA_G, "ALPHA_H": ALPHA_H,
                    "N_BOOT_G": N_BOOT_G, "N_BOOT_H": N_BOOT_H},
       "gates": {"tn3k_train_pool": len(tn3k_train),
                 "tn5000_splits": {s: len(d) for s, d in SPLITS.items()},
                 "seg_qc": SEGQC, "erm_sanity": float(erm_chk), "r1p_integrity": float(r1p_chk)},
       "arms_meta": {m: {"best_val_auroc": float(ARMS[m]["best_val_auroc"]),
                         "tn3k_val_dice_saved": float(ARMS[m]["tn3k_val_dice_saved"]),
                         "seg_feasible_any": bool(ARMS[m]["seg_feasible_any"]),
                         "n_joint_feasible": int(ARMS[m]["n_joint_feasible"])} for m in ARMS},
       "best_val_auroc": {m: float(ARMS[m]["best_val_auroc"]) for m in ARMS},
       "familyG": FAMG.to_dict(orient="records"),
       "familyH": FAMH.to_dict(orient="records"),
       "ext_seg_qc": {name: EXT[name]["JOINTg_qc"] for name in EXT}}
with open(OUTPUT_ROOT / "run_summary.json", "w") as f:
    json.dump(run, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Artifacts in", OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.iterdir()):
    print(f"  {p.name:38s} {p.stat().st_size/1024:9.1f} KB")
print("\nSave joint_g_trunk.pt / joint_img_trunk.pt + all JSON/CSV as a Kaggle dataset for Phases 5-7.")